# TimeSeries

`TimeSeries` is a metadata-rich container for a single time series. It carries your data together with its context — name, unit, frequency, timezone, location — as one self-describing object, fully interoperable with pandas, NumPy, Polars, and PyArrow.

Key characteristics:

- **Four data shapes**: SIMPLE, VERSIONED, CORRECTED, AUDIT — matching the read patterns of a bi-temporal database
- **Rich metadata**: name, unit, data type, geographic location, timezone, frequency, and arbitrary labels
- **Format interop**: `from_pandas()` / `to_pandas()`, `to_polars()`, `to_numpy()`, `to_pyarrow()`
- **Unit conversion**: `convert_unit()` for value scaling via pint

In [ ]:
import pandas as pd
import polars as pl

from timedatamodel import TimeSeries, DataType, Frequency, TimeSeriesType, GeoLocation

## Quick start — SIMPLE shape

The most common shape is **SIMPLE**: one observation per `valid_time`. Use `TimeSeries.from_pandas()` to create one from a DataFrame.

In [ ]:
df_simple = pd.DataFrame({
    "valid_time": pd.date_range("2024-01-15", periods=24, freq="1h", tz="UTC"),
    "value": [
        120.0, 115.0, 108.0, 105.0, 102.0, 100.0,
        110.0, 135.0, 160.0, 175.0, 180.0, 178.0,
        172.0, 170.0, 168.0, 165.0, 175.0, 190.0,
        200.0, 195.0, 180.0, 165.0, 145.0, 130.0,
    ],
})

ts = TimeSeries.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
    location=GeoLocation(latitude=57.7, longitude=11.97),
    timezone="Europe/Stockholm",
    description="Hourly wind power output from farm Alpha",
)
ts

The `valid_time` column can also live in the DataFrame index — `from_pandas()` handles both layouts:

In [ ]:
df_indexed = df_simple.set_index("valid_time")
ts_from_index = TimeSeries.from_pandas(df_indexed, name="wind_power", unit="MW")

print(f"Shape: {ts_from_index.shape}")
print(f"Rows:  {ts_from_index.num_rows}")

## Key properties

| Property | Description |
|---|---|
| `shape` | `DataShape` enum — which temporal columns are present |
| `num_rows` | Number of rows in the Polars DataFrame |
| `columns` | List of column names |
| `df` | The underlying Polars DataFrame (read-only by convention) |
| `has_missing` | `True` if the `value` column contains any nulls |

In [ ]:
print(f"shape:       {ts.shape}")
print(f"num_rows:    {ts.num_rows}")
print(f"columns:     {ts.columns}")
print(f"len():       {len(ts)}")
print(f"has_missing: {ts.has_missing}")
print()
print("Underlying Polars schema:")
print(ts.df.schema)

## Creating from a Polars DataFrame

`TimeSeries.from_polars()` wraps an existing Polars DataFrame directly — useful when data is already in Polars format (e.g. fetched from a database via `connectorx` or `adbc`). All timestamp columns must use `pl.Datetime("us", time_zone="UTC")`.

In [ ]:
pl_df = pl.DataFrame({
    "valid_time": pd.date_range("2024-01-15", periods=6, freq="1h", tz="UTC"),
    "value": [100.0, 105.0, None, 108.0, 103.0, 98.0],
}).with_columns(
    pl.col("valid_time").cast(pl.Datetime("us", time_zone="UTC"))
)

ts_from_polars = TimeSeries.from_polars(
    pl_df,
    name="load",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
)
print(f"has_missing: {ts_from_polars.has_missing}")
ts_from_polars

## The four data shapes

`DataShape` is inferred automatically from which timestamp columns are present in the DataFrame.

| Shape | Temporal columns | Use case |
|---|---|---|
| **SIMPLE** | `valid_time` | Flat, immutable observations |
| **VERSIONED** | `knowledge_time`, `valid_time` | Forecast revision history |
| **CORRECTED** | `valid_time`, `change_time` | Corrected meter readings (no forecast) |
| **AUDIT** | `knowledge_time`, `change_time`, `valid_time` | Full bi-temporal log |

### VERSIONED shape

A `VERSIONED` series stores the full forecast revision history — one row per `(knowledge_time, valid_time)` pair. This is the standard output of a forecast-issuing system where new runs update future values.

In [ ]:
# Two forecast runs (knowledge_times), each covering 6 hours ahead
rows = []
for kt_offset, values in [
    (0, [100.0, 105.0, 110.0, 108.0, 103.0, 98.0]),
    (6, [102.0, 107.0, 112.0, 111.0, 106.0, 101.0]),
]:
    kt = pd.Timestamp("2024-01-15 00:00", tz="UTC") + pd.Timedelta(hours=kt_offset)
    for h, v in enumerate(values):
        rows.append({
            "knowledge_time": kt,
            "valid_time": pd.Timestamp("2024-01-15 00:00", tz="UTC") + pd.Timedelta(hours=h),
            "value": v,
        })

df_versioned = pd.DataFrame(rows)

ts_v = TimeSeries.from_pandas(
    df_versioned,
    name="wind_power_forecast",
    unit="MW",
    frequency=Frequency.PT1H,
    data_type=DataType.FORECAST,
    timeseries_type=TimeSeriesType.OVERLAPPING,
)
ts_v

### CORRECTED shape

A `CORRECTED` series captures historical corrections to flat (non-forecast) data — for example, when sensor readings are recalibrated. Each correction adds a `change_time` column recording when the edit was made. CORRECTED shapes can only be created via `from_polars()` (read from a database), not `from_pandas()`.

In [ ]:
corrected_df = pl.DataFrame({
    "valid_time":  [
        "2024-01-15 00:00:00",
        "2024-01-15 01:00:00",
        "2024-01-15 01:00:00",  # second row is corrected
    ],
    "change_time": [
        "2024-01-20 09:00:00",
        "2024-01-20 09:00:00",
        "2024-01-22 14:30:00",  # later correction
    ],
    "value": [120.0, 115.0, 116.5],  # 116.5 corrects 115.0
}).with_columns([
    pl.col("valid_time").str.to_datetime().dt.replace_time_zone("UTC"),
    pl.col("change_time").str.to_datetime().dt.replace_time_zone("UTC"),
])

ts_c = TimeSeries.from_polars(
    corrected_df,
    name="meter_reading",
    unit="MWh",
    data_type=DataType.OBSERVATION,
)
ts_c

## Data access — head, tail, has_missing

`head()` and `tail()` return a new `TimeSeries` with the first or last *n* rows, preserving all metadata. `has_missing` is a quick boolean check for null values in the `value` column.

In [ ]:
print(f"has_missing: {ts.has_missing}")
print()
print("head(3):")
print(ts.head(3).df)
print()
print("tail(3):")
print(ts.tail(3).df)

## Coverage bar

`coverage_bar()` returns a `CoverageBar` object that renders as an SVG in Jupyter — each bin is green when at least one value in that bin is present, and gray when all values are null.

In [ ]:
# ts has no missing values — fully filled bar
ts.coverage_bar()

In [ ]:
# ts_from_polars has one null — the gap shows up as a gray bin
ts_from_polars.coverage_bar()

## Unit conversion

`convert_unit()` returns a new `TimeSeries` with values scaled to the target unit using [pint](https://pint.readthedocs.io/). The `unit` metadata field is updated automatically — all other metadata is preserved.

In [ ]:
ts_mw = TimeSeries.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
)

ts_gw = ts_mw.convert_unit("GW")

print(f"Original:  unit={ts_mw.unit!r},  first value = {ts_mw.df['value'][0]:.1f}")
print(f"Converted: unit={ts_gw.unit!r}, first value = {ts_gw.df['value'][0]:.4f}")

## Converting to pandas

`to_pandas()` restores the conventional index used by the existing read API. The index structure depends on the shape:

| Shape | Index |
|---|---|
| SIMPLE | `valid_time` |
| VERSIONED | `(knowledge_time, valid_time)` MultiIndex |
| CORRECTED | `(valid_time, change_time)` MultiIndex |
| AUDIT | `(knowledge_time, change_time, valid_time)` MultiIndex |

In [ ]:
df_out = ts.to_pandas()
print(f"Index name: {df_out.index.name}")
df_out.head()

In [ ]:
df_v_out = ts_v.to_pandas()
print(f"Index names: {df_v_out.index.names}")
df_v_out.head(8)

## Other conversion methods

`to_polars()`, `to_list()`, `to_numpy()`, and `to_pyarrow()` export the series to other formats.
The matching `from_list()`, `from_numpy()`, and `from_pyarrow()` constructors complete the round-trip.

| Method | Direction | Extra dependency |
|---|---|---|
| `to_polars()` | TimeSeries → `pl.DataFrame` | none |
| `to_list()` | TimeSeries → `dict[str, list]` | none |
| `to_numpy()` | TimeSeries → `dict[str, np.ndarray]` | `numpy` |
| `to_pyarrow()` | TimeSeries → `pa.Table` | `pyarrow` |
| `from_list(data)` | `dict[str, list]` → TimeSeries | none |
| `from_numpy(data)` | `dict[str, np.ndarray]` → TimeSeries | `numpy` |
| `from_pyarrow(table)` | `pa.Table` → TimeSeries | `pyarrow` |

In [ ]:
# No extra dependencies needed
print("to_polars():", type(ts.to_polars()))
print()
cols = ts.to_list()
print("to_list() keys:", list(cols.keys()))
print("to_list() first value:", cols["value"][0])
print()

# Requires numpy
import numpy as np
arr = ts.to_numpy()
print("to_numpy() keys:", list(arr.keys()))
print("to_numpy() values dtype:", arr["value"].dtype)
print("to_numpy() first value:", arr["value"][0])
print()

# Requires pyarrow
import pyarrow as pa
tbl = ts.to_pyarrow()
print("to_pyarrow() schema:", tbl.schema)

## Constructing from other formats

`from_list()`, `from_numpy()`, and `from_pyarrow()` are the exact inverses — metadata is preserved across each roundtrip.

In [ ]:
# Roundtrip via list
ts_from_list = TimeSeries.from_list(ts.to_list(), name=ts.name, unit=ts.unit, frequency=ts.frequency)
print("from_list() rows:", ts_from_list.num_rows)

# Roundtrip via numpy
ts_from_numpy = TimeSeries.from_numpy(ts.to_numpy(), name=ts.name, unit=ts.unit, frequency=ts.frequency)
print("from_numpy() rows:", ts_from_numpy.num_rows)

# Roundtrip via pyarrow
ts_from_arrow = TimeSeries.from_pyarrow(ts.to_pyarrow(), name=ts.name, unit=ts.unit, frequency=ts.frequency)
print("from_pyarrow() rows:", ts_from_arrow.num_rows)

assert ts_from_list.df.equals(ts.df)
assert ts_from_numpy.df.equals(ts.df)
assert ts_from_arrow.df.equals(ts.df)
print("All roundtrips verified.")

## Metadata

`metadata_dict()` returns all metadata fields as a plain dict — useful for serialisation or logging.

In [ ]:
import json
print(json.dumps(ts.metadata_dict(), indent=2, default=str))

## Labels

Arbitrary key-value labels let you tag series for filtering and provenance. They appear in the repr when set.

In [ ]:
ts_labelled = TimeSeries.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency=Frequency.PT1H,
    labels={"site": "Alpha", "turbine_model": "V150-4.5", "country": "SE"},
)
ts_labelled

## Summary

In this notebook you learned how to:

- Create a `TimeSeries` from a pandas DataFrame (`from_pandas`) or a Polars DataFrame (`from_polars`)
- Understand the four **data shapes** (SIMPLE, VERSIONED, CORRECTED, AUDIT) and when each is used
- Inspect key properties: `shape`, `num_rows`, `columns`, `df`, `has_missing`
- Access first/last rows with `head()` and `tail()`
- Visualise value coverage with `coverage_bar()` — SVG in Jupyter, Unicode blocks in terminal
- Convert values to a different unit with `convert_unit()` (pint-powered)
- Convert to different formats with `to_pandas()`, `to_polars()`, `to_list()`, `to_numpy()`, and `to_pyarrow()`
- Attach metadata: name, unit, data type, location, timezone, frequency, and arbitrary labels